# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
!git clone "https://github.com/FatimaNdeem/Flyrank-ml-internship." /content/Flyrank-ml-internship.

fatal: destination path '/content/Flyrank-ml-internship.' already exists and is not an empty directory.


In [13]:
%cd /content/Flyrank-ml-internship.

import os

print("Current directory:", os.getcwd())
print("\nRepository contents:")
!ls -lh

print("\nData folder:")
!ls -lh data/

/content/Flyrank-ml-internship.
Current directory: /content/Flyrank-ml-internship.

Repository contents:
total 84K
-rw-r--r--  1 root root  654 Aug 22 00:11 AGENTS.md
-rw-r--r--  1 root root  654 Aug 22 00:11 CLAUDE.md
drwxr-xr-x  3 root root 4.0K Aug 22 00:11 data
-rw-r--r--  1 root root 2.7K Aug 22 00:11 DATA_USE.md
drwxr-xr-x  2 root root 4.0K Aug 22 00:11 docs
-rw-r--r--  1 root root  11K Aug 22 00:11 GUIDE.md
-rw-r--r--  1 root root 1.3K Aug 22 00:11 LICENSE
drwxr-xr-x  2 root root 4.0K Aug 22 00:11 notebooks
drwxr-xr-x  3 root root 4.0K Aug 22 00:11 outputs
-rw-r--r--  1 root root 9.6K Aug 22 00:11 README.md
-rw-r--r--  1 root root  107 Aug 22 00:11 requirements.txt
drwxr-xr-x  2 root root 4.0K Aug 22 00:11 scripts
-rw-r--r--  1 root root 5.6K Aug 22 00:11 SETUP.md
drwxr-xr-x 14 root root 4.0K Aug 22 00:11 skills
drwxr-xr-x  2 root root 4.0K Aug 22 00:11 submission
drwxr-xr-x  4 root root 4.0K Aug 22 00:19 work

Data folder:
total 4.0K
drwxr-xr-x 2 root root 4.0K Aug 22 00:11 raw

In [14]:
import os

print("Baseline queue exists:",
      os.path.exists("data/processed/baseline_refresh_queue.csv"))

print("Processed folder exists:",
      os.path.exists("data/processed"))

print("Work folder exists:",
      os.path.exists("work"))

Baseline queue exists: False
Processed folder exists: False
Work folder exists: True


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The action queue ranks content by observed decline risk and gives each item a human-readable reason for review. Items with stronger decline signals are placed higher in the queue, but the ranking is a prioritization aid rather than an automatic instruction to refresh content.

The main action is `REVIEW_REFRESH`, meaning the page should be reviewed by a content or SEO specialist before any change is made. Items without sufficient evidence for action remain under `KEEP_MONITORING`.

Reason codes make the ranking understandable: they indicate the observed signal behind the recommendation rather than claiming that the model proves a page needs to be changed.

In [15]:
import pandas as pd
import numpy as np
import os

# Load the anonymized starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# 1. Create the same decline target used in ML-08
# ---------------------------------------------------------

df["is_declining"] = (
    df["impressions_last_30d"]
    < 0.8 * df["impressions_prev_30d"]
).astype(int)

# ---------------------------------------------------------
# 2. Calculate observed impression change
# ---------------------------------------------------------

df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (
        (df["impressions_last_30d"] - df["impressions_prev_30d"])
        / df["impressions_prev_30d"]
    ) * 100,
    np.nan
)

# ---------------------------------------------------------
# 3. Assign action labels
# ---------------------------------------------------------

df["action_label"] = np.where(
    df["is_declining"] == 1,
    "REVIEW_REFRESH",
    "KEEP_MONITORING"
)

# ---------------------------------------------------------
# 4. Assign human-readable reason codes
# ---------------------------------------------------------

def get_reason_code(row):

    if row["is_declining"] == 1:

        # Declining + recently stale content
        if (
            pd.notna(row["days_since_last_update"])
            and row["days_since_last_update"] >= 180
        ):
            return "DECLINING_AND_STALE"

        # Declining + older content
        if (
            pd.notna(row["content_age_days"])
            and row["content_age_days"] >= 365
        ):
            return "DECLINING_OLDER_CONTENT"

        # Declining + weaker search position
        if (
            pd.notna(row["avg_position"])
            and row["avg_position"] > 20
        ):
            return "DECLINING_LOW_POSITION"

        # Declining based primarily on impressions
        return "DECLINING_IMPRESSIONS"

    return "KEEP_MONITORING"


df["reason_code"] = df.apply(
    get_reason_code,
    axis=1
)

# ---------------------------------------------------------
# 5. Create explicit action ordering
# ---------------------------------------------------------

# REVIEW_REFRESH items must appear before KEEP_MONITORING
df["action_order"] = np.where(
    df["action_label"] == "REVIEW_REFRESH",
    0,
    1
)

# ---------------------------------------------------------
# 6. Sort the action queue
# ---------------------------------------------------------

# Within REVIEW_REFRESH:
# largest observed impression declines come first.
#
# Within KEEP_MONITORING:
# items remain after all review items.

df = df.sort_values(
    by=[
        "action_order",
        "impression_change_pct"
    ],
    ascending=[
        True,
        True
    ],
    na_position="last"
)

# ---------------------------------------------------------
# 7. Select columns for the human-readable queue
# ---------------------------------------------------------

queue_columns = [
    "content_id",
    "action_label",
    "reason_code",
    "impression_change_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "avg_position"
]

queue = df[queue_columns].copy()

# ---------------------------------------------------------
# 8. Print validation checks
# ---------------------------------------------------------

print("\nAction counts:")
print(df["action_label"].value_counts())

print("\nReason-code counts:")
print(df["reason_code"].value_counts())

print("\nTop 10 action items:")
display(queue.head(10))

Dataset shape: (30000, 44)

Action counts:
action_label
REVIEW_REFRESH     16262
KEEP_MONITORING    13738
Name: count, dtype: int64

Reason-code counts:
reason_code
KEEP_MONITORING            13738
DECLINING_IMPRESSIONS       9727
DECLINING_LOW_POSITION      3745
DECLINING_OLDER_CONTENT     2708
DECLINING_AND_STALE           82
Name: count, dtype: int64

Top 10 action items:


,content_id,action_label,reason_code,impression_change_pct,impressions_last_30d,impressions_prev_30d,days_since_last_update,content_age_days,avg_position
23,content_2da6ae9d0882,REVIEW_REFRESH,DECLINING_OLDER_CONTENT,-100.0,0,34,20,502,13.9
39,content_4595e8704e07,REVIEW_REFRESH,DECLINING_LOW_POSITION,-100.0,0,4,104,348,36.3
48,content_326fa2fa449f,REVIEW_REFRESH,DECLINING_IMPRESSIONS,-100.0,0,3,1,91,8.3
49,content_f0717373e86e,REVIEW_REFRESH,DECLINING_IMPRESSIONS,-100.0,0,1,8,174,10.1
51,content_d8a23b5e10c5,REVIEW_REFRESH,DECLINING_IMPRESSIONS,-100.0,0,1,8,126,7.5
67,content_9042a5355ff7,REVIEW_REFRESH,DECLINING_IMPRESSIONS,-100.0,0,10,8,91,5.0
68,content_6bc2ec5f6061,REVIEW_REFRESH,DECLINING_LOW_POSITION,-100.0,0,124,8,125,23.0
111,content_46eaff5b4ae8,REVIEW_REFRESH,DECLINING_OLDER_CONTENT,-100.0,0,3,22,441,16.3
130,content_27682a7cd588,REVIEW_REFRESH,DECLINING_IMPRESSIONS,-100.0,0,11,8,90,9.4
141,content_0af426466565,REVIEW_REFRESH,DECLINING_IMPRESSIONS,-100.0,0,8,8,91,3.6


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended for SEO and content teams as a decision-support tool for prioritizing pages that may need closer review. It uses observed impression decline and supporting content signals to organize a review queue.

A `REVIEW_REFRESH` recommendation means that a human should investigate the page before deciding whether a refresh is appropriate. It does not mean that the page must be refreshed.

The playbook is based on a 30,000-row anonymized starter dataset and the observed patterns in that sample. It should not be treated as proof that a page will recover after being updated, and the recommendations may not generalize to the full FlyRank warehouse or to different search environments.

The playbook should therefore support human prioritization rather than replace SEO or editorial judgment.

In [16]:
# Section 2 — Intended use and limits

print("Dataset rows:", len(df))
print("Dataset columns:", len(df.columns))

print("\nAction distribution:")
print(
    df["action_label"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

review_count = (df["action_label"] == "REVIEW_REFRESH").sum()
monitor_count = (df["action_label"] == "KEEP_MONITORING").sum()

print("\nReview items:", review_count)
print("Monitoring items:", monitor_count)

print(
    "\nThe queue is intended for human decision-support, "
    "not automatic content changes."
)

Dataset rows: 30000
Dataset columns: 49

Action distribution:
            action  count
0   REVIEW_REFRESH  16262
1  KEEP_MONITORING  13738

Review items: 16262
Monitoring items: 13738

The queue is intended for human decision-support, not automatic content changes.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before acting on a `REVIEW_REFRESH` recommendation, a content or SEO specialist should review the page's search intent, current content quality, relevance, recent updates, and whether the observed decline is large enough to justify action. The reviewer should also consider whether the decline could be caused by factors outside the page itself, such as changes in search demand or the wider search environment.

The playbook should never automatically publish, rewrite, delete, or redirect content. It should not automatically change titles, headings, links, structured data, or other SEO elements. It should also not treat a model or reason code as proof that a content change will improve performance.

The final decision remains with a human reviewer who can inspect context that is not represented in the dataset.

In [17]:
# Section 3 — Human review checks and no-go controls

required_review_checks = [
    "Search intent still matches the page",
    "Content is relevant and factually appropriate",
    "Observed impression decline is meaningful",
    "Recent content changes have been considered",
    "Possible external/search-demand factors have been considered",
    "Page quality and depth have been reviewed"
]

no_go_actions = [
    "Automatically publish content changes",
    "Automatically delete a page",
    "Automatically redirect a page",
    "Automatically rewrite titles or headings",
    "Automatically modify SEO elements",
    "Automatically treat a prediction as proof of improvement"
]

print("Human review checks:")
for check in required_review_checks:
    print("-", check)

print("\nNo-go automated actions:")
for action in no_go_actions:
    print("-", action)

print("\nTotal review checks:", len(required_review_checks))
print("Total no-go actions:", len(no_go_actions))

Human review checks:
- Search intent still matches the page
- Content is relevant and factually appropriate
- Observed impression decline is meaningful
- Recent content changes have been considered
- Possible external/search-demand factors have been considered
- Page quality and depth have been reviewed

No-go automated actions:
- Automatically publish content changes
- Automatically delete a page
- Automatically redirect a page
- Automatically rewrite titles or headings
- Automatically modify SEO elements
- Automatically treat a prediction as proof of improvement

Total review checks: 6
Total no-go actions: 6


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The playbook should be monitored because search behavior, content patterns, and the underlying data can change over time. Recommendations should be reviewed if the observed decline rate changes substantially, if model performance falls, or if the feature distributions become different from the data used to build the model.

A retraining or validation review should be considered when new labeled data becomes available, when precision or recall declines meaningfully, or when the relationship between the input features and decline behavior changes.

These triggers do not automatically mean that the model is wrong. They indicate that the recommendations should be investigated and the model or thresholds may need to be revalidated.

In [18]:
# Section 4 — Monitoring / retrain triggers

decline_rate = df["is_declining"].mean()

print("Current decline rate:", round(decline_rate, 4))
print("Current decline count:", int(df["is_declining"].sum()))
print("Current non-decline count:", int((df["is_declining"] == 0).sum()))

monitoring_triggers = {
    "new_labeled_data": "Review and revalidate when new labeled outcomes become available.",
    "performance_drop": "Review if precision, recall, or F1 falls meaningfully on a new validation sample.",
    "feature_drift": "Review if important feature distributions change substantially.",
    "decline_rate_shift": "Review if the observed decline rate changes substantially.",
    "search_environment_change": "Review after major changes in search behavior or measurement conditions."
}

print("\nMonitoring / retrain triggers:")

for trigger, description in monitoring_triggers.items():
    print(f"- {trigger}: {description}")

print(
    "\nCurrent recommendation:",
    "Continue monitoring; no automatic retraining is triggered by this sample alone."
)

Current decline rate: 0.5421
Current decline count: 16262
Current non-decline count: 13738

Monitoring / retrain triggers:
- new_labeled_data: Review and revalidate when new labeled outcomes become available.
- performance_drop: Review if precision, recall, or F1 falls meaningfully on a new validation sample.
- feature_drift: Review if important feature distributions change substantially.
- decline_rate_shift: Review if the observed decline rate changes substantially.
- search_environment_change: Review after major changes in search behavior or measurement conditions.

Current recommendation: Continue monitoring; no automatic retraining is triggered by this sample alone.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The final ranked action queue is exported to `work/outputs/` so that it can be reused in the capstone paper. The export contains the observed decline signal, action label, reason code, and supporting fields needed to explain why an item was prioritized.

The exported queue is a decision-support artifact. It does not represent a list of pages that must be refreshed.

In [19]:
# Section 5 — Export the action queue for the paper

import os

# Create the output directory
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Export the complete ranked queue
queue_path = os.path.join(
    output_dir,
    "content_action_queue.csv"
)

queue.to_csv(
    queue_path,
    index=False
)

# Export a smaller summary of action counts
action_summary = (
    df["action_label"]
    .value_counts()
    .rename_axis("action_label")
    .reset_index(name="count")
)

summary_path = os.path.join(
    output_dir,
    "action_summary.csv"
)

action_summary.to_csv(
    summary_path,
    index=False
)

print("Exported files:")
print("-", queue_path)
print("-", summary_path)

print("\nQueue rows:", len(queue))
print("Queue columns:", len(queue.columns))

print("\nAction summary:")
display(action_summary)

Exported files:
- work/outputs/content_action_queue.csv
- work/outputs/action_summary.csv

Queue rows: 30000
Queue columns: 9

Action summary:


,action_label,count
0,REVIEW_REFRESH,16262
1,KEEP_MONITORING,13738


In [20]:
print("\nOutput directory contents:")
!ls -lh work/outputs/


Output directory contents:
total 2.6M
-rw-r--r-- 1 root root   62 Aug 22 00:20 action_summary.csv
-rw-r--r-- 1 root root 2.6M Aug 22 00:20 content_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.